# Step 1: Data Generation
This notebook generates simulated energy, weather, and occupancy data for the Smart Campus.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Setup Paths
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)

# Configuration
NUM_BUILDINGS = 5
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 2, 1)
HOURS = (END_DATE - START_DATE).total_seconds() // 3600
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='h')[:-1]
n_timestamps = len(date_range)

# 1. Generate Metadata
building_types = ['Academic', 'Hostel', 'Office']
buildings = []
for i in range(1, NUM_BUILDINGS + 1):
    b_type = building_types[i % len(building_types)]
    if b_type == 'Academic': area = 3000
    elif b_type == 'Hostel': area = 5000
    else: area = 1500
    buildings.append({
        'building_id': f'B{i:03d}',
        'building_type': b_type,
        'area_sq_m': area
    })
pd.DataFrame(buildings).to_csv(f'{data_dir}/buildings.csv', index=False)
print('Buildings generated.')

# 2. Weather Data
base_temp = 24
temps = base_temp + 5 * np.sin((date_range.hour - 6) * 2 * np.pi / 24) + np.random.normal(0, 1, n_timestamps)
df_weather = pd.DataFrame({'timestamp': date_range, 'temperature': temps, 'humidity': 50})
df_weather.to_csv(f'{data_dir}/weather_data.csv', index=False)
print('Weather generated.')

# 3. Occupancy & Energy
readings = []
occ_data = []

for b in buildings:
    for ts, temp in zip(date_range, temps):
        hour = ts.hour
        is_weekend = ts.weekday() >= 5
        
        # Logic
        occ = 'Low'
        if b['building_type'] == 'Office' and 9 <= hour <= 18 and not is_weekend: occ = 'High'
        elif b['building_type'] == 'Hostel' and (hour < 8 or hour > 20): occ = 'High'
        
        occ_data.append({'timestamp': ts, 'building_id': b['building_id'], 'occupancy_level': occ})
        
        # Energy Consumption
        base_load = b['area_sq_m'] * 0.002
        occ_factor = 1.5 if occ == 'High' else 0.5
        temp_factor = 1 + (max(0, temp - 24) * 0.1)
        kwh = base_load * occ_factor * temp_factor * np.random.normal(1, 0.05)
        
        readings.append({'timestamp': ts, 'building_id': b['building_id'], 'energy_kwh': round(kwh, 2)})


pd.DataFrame(occ_data).to_csv(f'{data_dir}/occupancy_data.csv', index=False)
pd.DataFrame(readings).to_csv(f'{data_dir}/energy_readings.csv', index=False)
print('Energy & Occupancy generated.')

Buildings generated.
Weather generated.
Energy & Occupancy generated.
